# FlightAI: Building an Airline Assistant with Tools

<div style="padding:16px;border-radius:12px;background:linear-gradient(135deg,#0f172a,#1d4ed8);color:white">
<strong>Learning objective</strong><br>
Build a conversational airline assistant that combines an OpenAI chat model, structured function tools, SQLite state, and a polished Gradio interface.
</div>

## What you will learn
1. How to configure an API-backed model safely.
2. How to describe a Python function as an LLM tool.
3. How a model requests a tool, and how the application executes it.
4. How to send tool results back so the model can write a natural answer.
5. How to keep reusable business data in SQLite.
6. How to expose the complete workflow through Gradio.

## 1. The architecture

```mermaid
flowchart LR
    A[Traveler] --> B[Gradio chat UI]
    B --> C[Conversation controller]
    C --> D[OpenAI chat model]
    D -->|tool call request| C
    C --> E[Python price tool]
    E --> F[(SQLite prices.db)]
    F --> E
    E --> C
    C --> D
    D -->|friendly final answer| B
```

**Important distinction:** the model does not run Python directly. It produces a structured request, and our application decides to execute the matching Python function.

## 2. Imports

We use `dotenv` for local secrets, `OpenAI` for chat completions, `sqlite3` for durable lookup data, `pathlib` for portable file paths, and `gradio` for the interface.

In [1]:
import os
import json
import sqlite3
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

c:\AI_ENGINEER_CORE\Agentic AI - Core Track\Day_06\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Configure credentials and model

`load_dotenv()` reads `.env` from the notebook directory. `OPENAI_MODEL` is a real configuration value, and `.env.example` documents the expected keys.

The code below prints only a masked key status—never the secret itself.

In [2]:
load_dotenv(override=True)

api_key = os.getenv("OPENAI_API_KEY")
model = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")

if not api_key:
    raise RuntimeError("OPENAI_API_KEY is missing. Copy .env.example to .env and add a valid key.")

print(f"API key loaded: {api_key[:7]}...{api_key[-4:]}")
print(f"Chat model: {model}")

client = OpenAI(api_key=api_key)

API key loaded: sk-proj...szgA
Chat model: gpt-4.1-mini


## 4. Create the airline database

A SQLite table makes the assistant's ticket prices reproducible and avoids embedding all state in the model prompt. We also seed several example fares.

`city TEXT PRIMARY KEY` ensures one current record per city. The `UPSERT` syntax updates an existing city or inserts a new one.

In [3]:
DB_PATH = Path.cwd() / "prices.db"

def initialize_database():
    with sqlite3.connect(DB_PATH) as connection:
        connection.execute(
            "CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL NOT NULL)"
        )
        connection.commit()

def set_ticket_price(city: str, price: float) -> str:
    normalized_city = city.strip().lower()
    if not normalized_city:
        raise ValueError("City cannot be empty")
    if price < 0:
        raise ValueError("Price cannot be negative")

    with sqlite3.connect(DB_PATH) as connection:
        connection.execute(
            """INSERT INTO prices (city, price) VALUES (?, ?)
               ON CONFLICT(city) DO UPDATE SET price = excluded.price""",
            (normalized_city, float(price)),
        )
        connection.commit()
    return f"Updated {city.title()}: ${price:.2f}"

initialize_database()

for city_name, fare in {
    "london": 799,
    "paris": 899,
    "tokyo": 1420,
    "sydney": 2999,
    "berlin": 499,
}.items():
    set_ticket_price(city_name, fare)

print(f"Database ready: {DB_PATH}")

Database ready: c:\AI_ENGINEER_CORE\Agentic AI - Core Track\Day_06\prices.db


## 5. Implement the Python tool

This function is the source of truth for prices. Normalizing the city to lowercase lets `London`, `london`, and `LONDON` map to the same SQLite key.

In [4]:
def get_ticket_price(destination_city: str) -> str:
    """Return the return-ticket price for a destination city."""
    print(f"DATABASE TOOL: destination_city={destination_city}", flush=True)

    city = destination_city.strip().lower()
    with sqlite3.connect(DB_PATH) as connection:
        row = connection.execute(
            "SELECT price FROM prices WHERE city = ?", (city,)
        ).fetchone()

    if row is None:
        return f"No price data is available for {destination_city.title()}."
    return f"The return ticket price to {destination_city.title()} is ${row[0]:.2f}."

print(get_ticket_price("London"))

DATABASE TOOL: destination_city=London
The return ticket price to London is $799.00.


## 6. Describe the tool to the model

This JSON Schema tells the model:
- what function is available,
- what the function does,
- which argument it accepts,
- and that `destination_city` is required.

The schema is documentation for the model; it does not execute the Python function.

In [5]:
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to a supported destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city the traveler wants to visit",
            }
        },
        "required": ["destination_city"],
        "additionalProperties": False,
    },
}

tools = [{"type": "function", "function": price_function}]
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to a supported destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city the traveler wants to visit'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

## 7. Convert tool requests into tool results

The model can ask for a tool call by name and include JSON arguments. The loop below parses each request, executes only known functions, and creates a result message with the matching `tool_call_id`.

In [6]:
def execute_tool_calls(message):
    tool_results = []

    for tool_call in message.tool_calls or []:
        name = tool_call.function.name
        try:
            arguments = json.loads(tool_call.function.arguments or "{}")
        except json.JSONDecodeError:
            arguments = {}

        if name == "get_ticket_price":
            city = arguments.get("destination_city", "")
            if not isinstance(city, str) or not city.strip():
                content = "A valid destination city is required."
            else:
                content = get_ticket_price(city)
        else:
            content = f"Unknown tool: {name}"

        tool_results.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": content,
        })

    return tool_results

## 8. Build the conversation loop

A chat turn works in this sequence:

1. Convert Gradio history into chat messages.
2. Ask the model for a response, exposing the tool schema.
3. If the model requests tools, execute them and append all results.
4. Ask the model again, using the tool results as evidence.
5. Stop after a natural-language response or a small safety limit.

In [7]:
SYSTEM_MESSAGE = """
You are FlightAI, a courteous airline customer-support assistant.
Answer in at most two short sentences.
Use the get_ticket_price tool for destination fare questions.
Never invent a price. If the tool reports missing data, say that clearly.
""".strip()

def normalize_history(history):
    normalized = []
    for item in history or []:
        if isinstance(item, dict):
            normalized.append({
                "role": item.get("role", "assistant"),
                "content": item.get("content", ""),
            })
        elif isinstance(item, (list, tuple)) and len(item) == 2:
            normalized.append({"role": "user", "content": str(item[0])})
            normalized.append({"role": "assistant", "content": str(item[1])})
    return normalized

def chat(message, history):
    messages = [{"role": "system", "content": SYSTEM_MESSAGE}]
    messages.extend(normalize_history(history))
    messages.append({"role": "user", "content": message})

    response = client.chat.completions.create(
        model=model,
        messages=messages,
        tools=tools,
    )

    tool_round = 0
    while response.choices[0].finish_reason == "tool_calls" and tool_round < 5:
        assistant_message = response.choices[0].message
        tool_results = execute_tool_calls(assistant_message)
        messages.append(assistant_message)
        messages.extend(tool_results)

        response = client.chat.completions.create(
            model=model,
            messages=messages,
            tools=tools,
        )
        tool_round += 1

    return response.choices[0].message.content or "I am sorry, I could not generate a response."

print(chat("How much is a return ticket to Paris?", []))

DATABASE TOOL: destination_city=Paris
A return ticket to Paris costs $899.00. Would you like information on anything else?


## 9. API health check

The print above verifies the complete tool loop. If it returned an error, read the exception before launching the UI.

In [8]:
health_response = client.chat.completions.create(
    model=model,
    messages=[{"role": "user", "content": "Reply with exactly: FlightAI is ready"}],
)
print(health_response.choices[0].message.content)

FlightAI is ready


## 10. Launch the polished assistant

Run this cell to open the chatbot. The interface includes a title, concise instructions, and example traveler questions.

In [9]:
demo = gr.ChatInterface(
    fn=chat,
    title="✈️ FlightAI Airline Assistant",
    description="Ask about a return-ticket price. FlightAI will use structured tools and verified SQLite data.",
    examples=[
        "How much is a return ticket to Paris?",
        "What does a London ticket cost?",
        "Do you have a price for Rome?",
        "Compare Tokyo and Sydney ticket prices",
    ],
)

demo.launch(prevent_thread_lock=True, inbrowser=False)

TypeError: ChatInterface.__init__() got an unexpected keyword argument 'type'

## 11. Summary and experiments

You have now seen the complete tool-calling loop:

- **Prompt:** asks for a destination fare.
- **Model:** requests `get_ticket_price`.
- **Application:** validates arguments and queries SQLite.
- **Tool result:** becomes evidence for the model.
- **Final response:** is courteous and grounded in data.

### Try these extensions
1. Add a second tool for baggage allowances.
2. Add a tool for changing ticket prices.
3. Record conversation analytics in a second SQLite table.
4. Add input validation for unsupported cities.
5. Stream model responses instead of returning them once.